Imports and paths

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PRICE_PATH = RAW_DATA_DIR / "price.csv"
NEWS_PATH = RAW_DATA_DIR / "news.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Price data found: {PRICE_PATH.exists()}")
print(f"News data found: {NEWS_PATH.exists()}")

Matplotlib is building the font cache; this may take a moment.


Project root: /Users/camilo/stock-news-forecasting
Price data found: True
News data found: True


Load data

In [2]:
prices_raw = pd.read_csv(
    PRICE_PATH,
    parse_dates=["date"],
)

news_raw = pd.read_csv(
    NEWS_PATH,
    parse_dates=["datetime"],
)

print(f"Price dataset: {prices_raw.shape[0]:,} rows × {prices_raw.shape[1]} columns")
print(f"News dataset:  {news_raw.shape[0]:,} rows × {news_raw.shape[1]} columns")

display(prices_raw.head())
display(news_raw.head())

Price dataset: 1,687 rows × 7 columns
News dataset:  4,440 rows × 4 columns


,date,ticker,open,high,low,close,volume
0,2023-11-13,AAPL,185.8200,186.0300,184.2100,184.8000,43627500
1,2023-11-14,AAPL,187.7000,188.1100,186.3000,187.4400,60108400
2,2023-11-15,AAPL,187.8500,189.5000,187.7800,188.0100,53790500
3,2023-11-16,AAPL,189.5700,190.9600,188.6500,189.7100,54412900
4,2023-11-17,AAPL,190.2500,190.3800,188.5700,189.6900,50922700


,datetime,ticker,headline,summary
0,2024-10-29 18:07:48,AAPL,Apple Unveils the Redesigned Mac Mini,-- Apple overhauled the design of its Mac mini...
1,2024-10-29 02:21:10,AAPL,Apple blocked from selling iPhone 16 in Indone...,TECH giant Apple will not be allowed to sell i...
2,2024-10-28 14:05:22,AAPL,"Apple Rises on Apple Intelligence Rollout, New...",-- Apple unveiled its new iMac and said Apple ...
3,2024-10-28 12:05:04,AAPL,Apple : How Apple developed the world’s first ...,apple stories Inside the Audio Lab: How Apple ...
4,2024-10-28 11:02:06,AAPL,Apple launches the iPhone into the AI era with...,Apple is releasing a free software update that...


Schema audit

In [3]:
def schema_report(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "non_null": df.notna().sum(),
            "null_count": df.isna().sum(),
            "null_pct": df.isna().mean().mul(100).round(2),
            "unique_values": df.nunique(dropna=False),
        }
    )


print("Price schema")
display(schema_report(prices_raw))

print("News schema")
display(schema_report(news_raw))

Price schema


,dtype,non_null,null_count,null_pct,unique_values
date,datetime64[us],1687,0,0.0000,241
ticker,str,1687,0,0.0000,7
open,float64,1687,0,0.0000,1614
high,float64,1687,0,0.0000,1623
low,float64,1687,0,0.0000,1631
close,float64,1687,0,0.0000,1632
volume,int64,1687,0,0.0000,1684


News schema


,dtype,non_null,null_count,null_pct,unique_values
datetime,datetime64[us],4440,0,0.0000,4395
ticker,str,4440,0,0.0000,7
headline,str,4440,0,0.0000,4214
summary,str,4439,1,0.0200,4263


Coverage by ticker

In [4]:
price_coverage = (
    prices_raw.groupby("ticker")
    .agg(
        observations=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        unique_dates=("date", "nunique"),
    )
    .sort_index()
)

news_coverage = (
    news_raw.groupby("ticker")
    .agg(
        articles=("datetime", "size"),
        first_article=("datetime", "min"),
        last_article=("datetime", "max"),
        active_dates=("datetime", lambda values: values.dt.date.nunique()),
    )
    .sort_index()
)

display(price_coverage)
display(news_coverage)

,observations,first_date,last_date,unique_dates
ticker,,,,
AAPL,241,2023-11-13,2024-10-28,241
AMZN,241,2023-11-13,2024-10-28,241
GOOGL,241,2023-11-13,2024-10-28,241
META,241,2023-11-13,2024-10-28,241
MSFT,241,2023-11-13,2024-10-28,241
NVDA,241,2023-11-13,2024-10-28,241
TSLA,241,2023-11-13,2024-10-28,241


,articles,first_article,last_article,active_dates
ticker,,,,
AAPL,799,2023-11-13 17:35:52,2024-10-29 18:07:48,224
AMZN,941,2023-11-13 12:38:25,2024-10-29 11:35:05,238
GOOGL,386,2023-11-14 12:07:58,2024-10-29 19:34:15,155
META,391,2023-11-16 15:34:02,2024-10-29 19:06:19,179
MSFT,535,2023-11-13 14:53:47,2024-10-29 12:42:25,193
NVDA,594,2023-11-14 05:35:10,2024-10-28 17:53:12,173
TSLA,794,2023-11-13 09:12:41,2024-10-29 16:59:40,225


Duplicate audit

In [5]:
duplicate_report = pd.DataFrame(
    {
        "check": [
            "Exact duplicate price rows",
            "Duplicate ticker-date price records",
            "Exact duplicate news rows",
            "Duplicate ticker-datetime news records",
            "Duplicate ticker-headline records",
        ],
        "count": [
            prices_raw.duplicated().sum(),
            prices_raw.duplicated(subset=["ticker", "date"]).sum(),
            news_raw.duplicated().sum(),
            news_raw.duplicated(subset=["ticker", "datetime"]).sum(),
            news_raw.duplicated(subset=["ticker", "headline"]).sum(),
        ],
    }
)

display(duplicate_report)

,check,count
0,Exact duplicate price rows,0
1,Duplicate ticker-date price records,0
2,Exact duplicate news rows,1
3,Duplicate ticker-datetime news records,35
4,Duplicate ticker-headline records,139


OHLCV integrity checks

In [6]:
price_integrity = pd.Series(
    {
        "missing_values": int(prices_raw.isna().sum().sum()),
        "duplicate_ticker_dates": int(
            prices_raw.duplicated(subset=["ticker", "date"]).sum()
        ),
        "non_positive_open": int((prices_raw["open"] <= 0).sum()),
        "non_positive_high": int((prices_raw["high"] <= 0).sum()),
        "non_positive_low": int((prices_raw["low"] <= 0).sum()),
        "non_positive_close": int((prices_raw["close"] <= 0).sum()),
        "negative_volume": int((prices_raw["volume"] < 0).sum()),
        "high_below_open": int(
            (prices_raw["high"] < prices_raw["open"]).sum()
        ),
        "high_below_close": int(
            (prices_raw["high"] < prices_raw["close"]).sum()
        ),
        "low_above_open": int(
            (prices_raw["low"] > prices_raw["open"]).sum()
        ),
        "low_above_close": int(
            (prices_raw["low"] > prices_raw["close"]).sum()
        ),
    },
    name="violations",
)

display(price_integrity.to_frame())

,violations
missing_values,0
duplicate_ticker_dates,0
non_positive_open,0
non_positive_high,0
non_positive_low,0
non_positive_close,0
negative_volume,0
high_below_open,0
high_below_close,0
low_above_open,0


Clean copies

In [7]:
prices = (
    prices_raw
    .copy()
    .assign(ticker=lambda df: df["ticker"].str.strip().str.upper())
    .sort_values(["ticker", "date"])
    .drop_duplicates(subset=["ticker", "date"])
    .reset_index(drop=True)
)

news = (
    news_raw
    .copy()
    .assign(
        ticker=lambda df: df["ticker"].str.strip().str.upper(),
        headline=lambda df: df["headline"].str.strip(),
        summary=lambda df: df["summary"].fillna("").str.strip(),
    )
    .sort_values(["ticker", "datetime"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Clean price shape: {prices.shape}")
print(f"Clean news shape:  {news.shape}")

Clean price shape: (1687, 7)
Clean news shape:  (4439, 4)
